# Lekcija 08 - Višestruki agentski dizajnerski obrazac


## Postavljanje


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Zašto sustavi s više agenata?

Zadaci iz stvarnog svijeta poput planiranja putovanja uključuju mnoge različite vrste stručnosti — logistiku, lokalno znanje, budžetiranje i još mnogo toga. Jedan agent koji pokušava upravljati svime brzo postaje nezgrapan.

Sustavi s više agenata rješavaju to kroz **specijalizaciju**: svaki agent se fokusira na jedno područje stručnosti, proizvodeći rezultate veće kvalitete nego generalist. Oni također poboljšavaju **skalabilnost** — možete dodati nove agente (npr. stručnjaka za letove, kritičara restorana) bez prepisivanja postojećeg tijeka rada. Agenti se povezuju kroz strukturirani tijek rada, prenoseći kontekst s jednog na drugog.


## Izrada specijaliziranih agenata


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Izgradnja sekvencijalnog tijeka rada

`WorkflowBuilder` vam omogućuje povezivanje agenata u usmjereni graf. Ovdje stvaramo jednostavnu dvostupanjski cjevovod: **TravelPlanner** izrađuje plan putovanja, a zatim ga **TravelConcierge** pregledava i poboljšava.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodavanje Više Agenata u Tijek Rada

Jedna od najvećih prednosti obrasca s više agenata je koliko je jednostavno proširiti ga. Dolje dodajemo agenta **BudgetReviewer** koji provjerava plan u odnosu na proračun putnika, označava stavke koje bi mogle premašiti troškove, te predlaže alternative za uštedu novca. Tijek rada sada pokreće tri agenta jedan za drugim:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Sažetak

U ovoj lekciji naučili ste kako:

1. **Stvoriti specijalizirane agente** — svaki s fokusiranom ulogom (planiranje, concierge, pregled proračuna).
2. **Povezati agente u sekvencijalni tijek rada** koristeći `WorkflowBuilder` i `add_edge`.
3. **Prikazivati izlaz u stvarnom vremenu** iz višestrukog agentskog procesa, prateći koji agent govori.
4. **Proširiti tijek rada** dodavanjem novih agenata u lanac bez mijenjanja postojećih.

Dizajn s više agenata održava svakog agenta jednostavnim, dok istovremeno proizvodi bogatije i temeljitije pregledane rezultate nego što bi to pojedinačni agent mogao postići samostalno.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:  
Ovaj je dokument preveden pomoću AI usluge za prevođenje [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba se smatrati službenim i autoritativnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne snosimo odgovornost za bilo kakva nesporazumevanja ili pogrešna tumačenja koja proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
